In [1]:
# ==============================================================================
# INITIALISATION ET IMPORTATION DES LIBRAIRIES
# ==============================================================================
import os
import re
import string
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import nltk

# --- Outils de Traitement du Langage (NLP) et Similarité ---
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.stem import WordNetLemmatizer
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from wordcloud import WordCloud

# --- Machine Learning ---
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

# --- Explainable AI ---
import shap

# --- Configuration visuelle et environnement ---
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
sns.set_theme(style="whitegrid")

# Téléchargement des dictionnaires NLTK
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('vader_lexicon', quiet=True)

print("✅ Cellule 1 terminée : Tous les outils sont chargés.")

✅ Cellule 1 terminée : Tous les outils sont chargés.


In [3]:
def auditer_et_corriger_dataset(dataframe):
    print("--- DÉBUT DE L'AUDIT DE QUALITÉ ---")
    df_propre = dataframe.copy()
    
    taille_avant = len(df_propre)
    df_propre = df_propre.dropna(subset=['text_', 'label'])
    if taille_avant - len(df_propre) > 0:
        print(f"🔧 CORRECTION : {taille_avant - len(df_propre)} lignes nulles supprimées.")

    taille_avant = len(df_propre)
    df_propre = df_propre[df_propre['label'].isin(['CG', 'OR'])]
    if taille_avant - len(df_propre) > 0:
        print(f"🔧 CORRECTION : {taille_avant - len(df_propre)} labels aberrants exclus.")

    taille_avant = len(df_propre)
    df_propre['text_'] = df_propre['text_'].replace(r'^\s*$', np.nan, regex=True)
    df_propre = df_propre.dropna(subset=['text_'])
    
    print(f"🟢 DATASET VALIDÉ : Taille finale = {len(df_propre)} lignes.")
    return df_propre

df = pd.read_csv('fake reviews dataset.csv')
df = auditer_et_corriger_dataset(df)
df = df.drop_duplicates().reset_index(drop=True)

--- DÉBUT DE L'AUDIT DE QUALITÉ ---
🟢 DATASET VALIDÉ : Taille finale = 40432 lignes.


In [1]:
print("\n--- GÉNÉRATION DU TABLEAU DE BORD (EDA) ---")
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
 
sns.countplot(data=df, x='label', ax=axes[0], palette=['#3498db', '#e74c3c'], hue='label', legend=False)
axes[0].set_title('Répartition des Avis (CG = Fraude, OR = Original)', fontweight='bold')
 
sns.countplot(data=df, x='rating', hue='label', ax=axes[1], palette=['#3498db', '#e74c3c'])
axes[1].set_title('Distribution des Notes', fontweight='bold')
 
df['longueur_mots'] = df['text_'].apply(lambda x: len(str(x).split()))
sns.histplot(data=df, x='longueur_mots', hue='label', bins=50, kde=True, ax=axes[2], palette=['#3498db', '#e74c3c'])
axes[2].set_title('Longueur des Avis (en mots)', fontweight='bold')
axes[2].set_xlim(0, 200)
 
plt.tight_layout()
plt.show()
 
df = df.drop(columns=['longueur_mots'])


--- GÉNÉRATION DU TABLEAU DE BORD (EDA) ---


NameError: name 'plt' is not defined

In [2]:
print("--- GÉNÉRATION DES NUAGES DE MOTS ---")
 
textes_fraude = " ".join(text for text in df[df['label'] == 'CG']['text_'].dropna())
textes_originaux = " ".join(text for text in df[df['label'] == 'OR']['text_'].dropna())
 
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
 
wordcloud_cg = WordCloud(width=800, height=400, background_color='white', colormap='Reds', max_words=100).generate(textes_fraude)
axes[0].imshow(wordcloud_cg, interpolation='bilinear')
axes[0].set_title('Mots les plus fréquents - FRAUDES (CG)', fontsize=14, fontweight='bold')
axes[0].axis('off')
 
wordcloud_or = WordCloud(width=800, height=400, background_color='white', colormap='Blues', max_words=100).generate(textes_originaux)
axes[1].imshow(wordcloud_or, interpolation='bilinear')
axes[1].set_title('Mots les plus fréquents - ORIGINAUX (OR)', fontsize=14, fontweight='bold')
axes[1].axis('off')
 
plt.tight_layout()
plt.show()

--- GÉNÉRATION DES NUAGES DE MOTS ---


NameError: name 'df' is not defined